# Optuna with Scikit‑Learn Pipelines

This notebook demonstrates end‑to‑end hyperparameter optimization using **Optuna** with a reusable scikit‑learn pipeline.

[Optuna on PyPi](https://pypi.org/project/optuna/)

### Where Optuna Fits 

- Dataset arrives, customer wants a model  
  - You perform best model and scaler workbook
  - Once model and scaler identified:
    - Run Optuna to identify best hyperparameters for your model  

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.compose import make_column_selector as selector
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression

import optuna
import optuna_dashboard

## 1. Imports

In [2]:
# All features are categorical (including integer-valued ones),
# so we encode every column with OrdinalEncoder then scale with StandardScaler.
# Using an explicit list of all columns avoids dtype-based selectors
# accidentally skipping integer-looking columns.
'''
ordinal_enc = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

preprocessor = make_column_transformer(
    (ordinal_enc, list(X.columns)),  # encode ALL columns
    remainder='drop',
)
'''

"\nordinal_enc = OrdinalEncoder(\n    handle_unknown='use_encoded_value',\n    unknown_value=-1\n)\n\npreprocessor = make_column_transformer(\n    (ordinal_enc, list(X.columns)),  # encode ALL columns\n    remainder='drop',\n)\n"

## 2. Load Sample Data

(Replace with your own dataset if needed.)

In [3]:
TRANSFORM_URL = 'https://raw.githubusercontent.com/Oldmage112/TI4-Project/refs/heads/main/PowerBI/Derived%20CSVs/Factions_Transform.csv'
WINNERS_URL   = 'https://raw.githubusercontent.com/Oldmage112/TI4-Project/refs/heads/main/PowerBI/Derived%20CSVs/Factions_Winners.csv'

df_transform = pd.read_csv(TRANSFORM_URL, index_col=-1)
df_winners   = pd.read_csv(WINNERS_URL, index_col=0)

# Merge on Index
df = df_transform.merge(df_winners, on='Index', how='inner')

# All columns are categorical — force every column to object dtype
# so dtype-based selectors never mis-classify integer-valued columns as numeric
df = df.astype(str)

target_col = 'WinningFaction'
X = df.drop(columns=[target_col])
y = df[target_col]

print(f'Shape: {X.shape}  |  Target classes: {y.unique()}')


Shape: (17797, 49)  |  Target classes: <StringArray>
[            'XxchaKingdom',            'KeleresMentak',
            'YssarilTribes',             'ArgentFlight',
          'Vuil'raithCabal',      'MahactGeneSorcerers',
           'GhostsofCreuss',                  'Arborec',
             'SardakkN'orr',             'L1Z1XMindnet',
               'TitansofUl',          'NaaluCollective',
          'EmiratesofHacan',               'NekroVirus',
           'YinBrotherhood',          'MentakCoalition',
        'NaazRokhaAlliance',                    'Nomad',
     'UniversitiesofJolNar',               'ClanofSaar',
          'FederationofSol',            'EmbersofMuaat',
                 'Empyrean',                    'Winnu',
             'KeleresXxcha',           'BaronyofLetnev',
            'KeleresArgent',                  'Keleres',
    'DeepwroughtScholarate',              'LastBastion',
                'Firmament',         'RalNelConsortium',
         'CrimsonRebellion',       

## 3. Preprocessing

In [4]:
import numpy as np
from sklearn.compose import make_column_transformer
from sklearn.compose import make_column_selector as selector
from sklearn.preprocessing import OrdinalEncoder


categorical_preprocessor = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

preprocessor = make_column_transformer(
    (
        categorical_preprocessor,
        selector(dtype_exclude=np.number),
    ),
    remainder="passthrough",
)

## 4. Baseline Pipeline

In [5]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler',       StandardScaler()),
    ('classifier',   LogisticRegression(random_state=42, max_iter=1000)),
])


## 5. Optuna Objective Function

In [6]:

def objective(trial):
    params = {
        # Regularisation strength (inverse): smaller = stronger regularisation
        'classifier__C': trial.suggest_float('C', 1e-4, 1e2, log=True),
        # Solver / penalty combinations supported by LogisticRegression
        'classifier__solver': trial.suggest_categorical('solver', ['lbfgs', 'saga']),
        'classifier__penalty': trial.suggest_categorical('penalty', ['l2', None]),
        'classifier__max_iter': trial.suggest_int('max_iter', 200, 2000),
    }

    # saga is the only solver that supports both l1 and l2;
    # lbfgs only supports l2 / None — prune incompatible combos early
    if params['classifier__solver'] == 'lbfgs' and params['classifier__penalty'] not in ('l2', None):
        raise optuna.TrialPruned()

    model_trial = clone(model)
    model_trial.set_params(**params)

    scores = cross_val_score(
        model_trial,
        X,
        y,
        cv=5,
        scoring='accuracy',
    )

    return scores.mean()


## 6. Run Optuna Study

In [7]:
%%time
study = optuna.create_study(direction="maximize",storage="sqlite:///db.sqlite3",  # Specify the storage URL here.
        study_name="quadratic-simple")
study.optimize(objective, n_trials=25)

study.best_params, study.best_value


[I 2026-04-12 20:12:41,589] A new study created in RDB with name: quadratic-simple
c:\Users\MGarn\miniconda3\envs\analyst-lab\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\MGarn\miniconda3\envs\analyst-lab\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\MGarn\miniconda3\envs\analyst-lab\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1

CPU times: total: 47min 56s
Wall time: 41min 16s


({'C': 0.0005935028055503585,
  'solver': 'saga',
  'penalty': 'l2',
  'max_iter': 1007},
 0.21200326123674432)

## 7. Train Final Model

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

best_model = clone(model)
best_model.set_params(**{
    f"classifier__{k}": v for k, v in study.best_params.items()
})

best_model.fit(X_train, y_train)
print(f'Best Model Score: {best_model.score(X_test, y_test):.3f}')
study.best_params


c:\Users\MGarn\miniconda3\envs\analyst-lab\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Best Model Score: 0.232


{'C': 0.0005935028055503585,
 'solver': 'saga',
 'penalty': 'l2',
 'max_iter': 1007}

In [9]:
best_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ordinalencoder', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [4]:
# Run in your terminal (not a Python cell):
# optuna-dashboard sqlite:///db.sqlite3
# Or uncomment the line below to launch from the notebook:
#!optuna-dashboard sqlite:///db.sqlite3
